# Resumable local-GPU experiment pipeline

This is the local equivalent of `experiment_pipeline.ipynb`. Each condition is saved to its own JSONL file under `gpu_experiments/pipeline_runs/`. Re-running skips successful rows; increasing `NUM_ROWS` processes only the new rows.

With `MAX_CONCURRENCY = None`, the pipeline tries every pending row concurrently and automatically halves the GPU batch on CUDA out-of-memory. The discovered safe level is reused across conditions.

In [ ]:
python -c "
from pathlib import Path
import importlib
import sys

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'content conditions').is_dir() and (candidate / 'gpu_experiments').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate the project root')

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import gpu_experiments.model_loader
import gpu_experiments.inference
import gpu_experiments.pipeline
importlib.reload(gpu_experiments.model_loader)
importlib.reload(gpu_experiments.inference)
importlib.reload(gpu_experiments.pipeline)
from gpu_experiments.model_loader import ModelConfig
from gpu_experiments.pipeline import run_pipeline
print('Project root:', REPO_ROOT)


MODEL = ModelConfig(
    model_id='Qwen/Qwen3-0.6B',
    dtype='bfloat16',
    device_map='auto',
    attention_implementation=None,  # Use 'flash_attention_2' only if installed
)
TARGET_CONDITIONS = list(range(7))  # 0-8 plus CoT baseline 20
NUM_ROWS = None                 # Per condition; None means all 448
START_ROW = 0
MAX_CONCURRENCY = None        # None = automatic maximum with OOM backoff
MAX_NEW_TOKENS = 1024
TEMPERATURE = 0.0
ENABLE_THINKING = False


result = run_pipeline(
    teacher_model='deepseek-v4-flash',
    model=MODEL,
    condition=TARGET_CONDITIONS,
    num_rows=NUM_ROWS,
    max_concurrency=MAX_CONCURRENCY,
    start_row=START_ROW,
    temperature=TEMPERATURE,
    max_new_tokens=MAX_NEW_TOKENS,
    enable_thinking=ENABLE_THINKING,
    retry_failed=True,
    repo_root=REPO_ROOT,
)
"

In [1]:
from pathlib import Path
import importlib
import sys

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'content conditions').is_dir() and (candidate / 'gpu_experiments').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate the project root')

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import gpu_experiments.model_loader
import gpu_experiments.inference
import gpu_experiments.pipeline
importlib.reload(gpu_experiments.model_loader)
importlib.reload(gpu_experiments.inference)
importlib.reload(gpu_experiments.pipeline)
from gpu_experiments.model_loader import ModelConfig
from gpu_experiments.pipeline import run_pipeline
print('Project root:', REPO_ROOT)

Project root: /home/f_goodarzi/run_qwen_gpqa/analogy-codex


## Settings

In [2]:
MODEL = ModelConfig(
    model_id='Qwen/Qwen3-0.6B',
    dtype='bfloat16',
    device_map='auto',
    attention_implementation=None,  # Use 'flash_attention_2' only if installed
)
TARGET_CONDITIONS = list(range(7))  # 0-8 plus CoT baseline 20
NUM_ROWS = None                 # Per condition; None means all 448
START_ROW = 0
MAX_CONCURRENCY = None        # None = automatic maximum with OOM backoff
MAX_NEW_TOKENS = 1024
TEMPERATURE = 0.0
ENABLE_THINKING = False

## Run or resume

This cell loads the model once, shares it across every condition, and flushes each completed GPU batch to disk.

In [4]:
result = run_pipeline(
    teacher_model='deepseek-v4-flash',
    model=MODEL,
    condition=TARGET_CONDITIONS,
    num_rows=NUM_ROWS,
    max_concurrency=MAX_CONCURRENCY,
    start_row=START_ROW,
    temperature=TEMPERATURE,
    max_new_tokens=MAX_NEW_TOKENS,
    enable_thinking=ENABLE_THINKING,
    retry_failed=True,
    repo_root=REPO_ROOT,
)
result

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Resume scan: 443 already successful; 5 row(s) remaining.


Local GPU inference:   0%|          | 0/5 [00:00<?, ?row/s]

Resume scan: 441 already successful; 7 row(s) remaining.


Local GPU inference:   0%|          | 0/7 [00:00<?, ?row/s]

KeyboardInterrupt: 